# CLAUDE API MECHANICS HOMEWORK

<br>

**About:** Practice applying Messages API, streaming, batches, and cost tracking on different prompts and datasets from the lesson.

**Learning Goals:**
- Build a budget-capped multi-prompt pipeline using MessagesClient.
- Use streaming to count tokens delivered in real time.
- Reason about realtime vs batch API tradeoffs for concrete scenarios.
- Submit and track a Message Batches API job over a real dataset.
- Combine API mechanics with domain-appropriate reasoning at scale.

**Keywords:** MessagesClient, budget, streaming, batches, cost, custom_id

**Prerequisite Knowledge:** Lesson 01 complete

**Target User:** Developer who has completed 01_claude_api_mechanics.ipynb


**Estimated Cost to Run:** < $0.05 in default mode

**Estimated Completion:** 30-45 min

<hr style="border: 4px solid#003262;" />



#### CONTENTS

- [Q1: Budget-Capped Multi-Prompt Pipeline](#q1-budget-capped-multi-prompt-pipeline)
- [Q2: Streaming with Token Counting](#q2-streaming-with-token-counting)
- [Q3: Realtime vs Batch Decision Justification](#q3-realtime-vs-batch-decision-justification)
- [Q4: Batches API Round-Trip over Wikipedia Snippets](#q4-batches-api-round-trip-over-wikipedia-snippets)

<br>

In [20]:
# Paths in this repo are written relative to the repository root, but a
# Jupyter kernel starts in the notebook's own directory. Anchor to the root.
from claude_code.paths import use_repo_root
use_repo_root()

# Setup - imports and environment check
import os
import time
import json
from pathlib import Path

import anthropic  # Verified against pypi.org/pypi/anthropic/json on 2026-09-02

from claude_code.api import MessagesClient, StreamingClient, BatchesClient, BatchRequest
from claude_code.api.errors import BudgetExceededError
from claude_code.config.model_registry import cost_for_call, get_model
from claude_code.security.secrets import load_api_key

DRY_RUN = os.getenv("CLAUDE_CODE_DRY_RUN", "0") == "1"
if DRY_RUN:
    print("DRY RUN: live API calls will be skipped")
    # Client objects need a key to construct; no request is ever sent in dry-run mode.
    os.environ.setdefault("ANTHROPIC_API_KEY", "dry-run-placeholder")
else:
    _ = load_api_key("ANTHROPIC_API_KEY")

<hr style="border: 2px solid#003262;" />

## Q1: Budget-Capped Multi-Prompt Pipeline

Write a function `run_prompts(prompts: list[str], budget_usd: float) -> list[str]` that:
- Accepts a list of prompts and a shared cost cap in USD.

- Uses [`MessagesClient`](https://platform.claude.com/docs/en/build-with-claude/working-with-messages) with `haiku_current`.
  - `client.messages.create()` is a standard blocking request. It waits for the model to finish generating the entire response before returning a single, complete Message object.

- Calls each prompt in order.

- Stops (without raising) if the next call would exceed `budget_usd`.

- Returns a list of response text strings (one per successful call).

Store the results of running the function on the sample prompts below in `results`.

In [13]:
# Q1: Budget-capped multi-prompt pipeline
SAMPLE_PROMPTS = [
    "Name one animal that lives in the ocean. One word.",
    "Name one planet in our solar system. One word.",
    "Name one programming language. One word.",
    "Name one country in South America. One word.",
]

haiku_current = "claude-haiku-4-5-20251001"

In [14]:
# Current pricing for Claude 4.5 Haiku per million tokens (as of late 2024 / 2026 conventions)
# Input: $0.80 / million tokens ($0.0000008 per token)
# Output: $4.00 / million tokens ($0.0000040 per token)
HAIKU_INPUT_COST_PER_TOKEN = 0.80 / 1_000_000
HAIKU_OUTPUT_COST_PER_TOKEN = 4.00 / 1_000_000

def run_prompts(prompts: list[str], budget_usd: float) -> list[str]:
    # Initialize the client (assumes ANTHROPIC_API_KEY is in the environment)
    client = anthropic.Anthropic()
    
    responses = []
    total_spent = 0.0

    for prompt in prompts:
        # Check if we have any budget remaining before attempting a call
        if total_spent >= budget_usd:
            break
            
        try:
            # Execute the call using MessagesClient (client.messages)
            message = client.messages.create(
                model=haiku_current, # Standard implementation mapping to 'haiku_current'
                max_tokens=50,
                messages=[
                    {"role": "user", "content": prompt}
                ]
            )
            
            # Extract token details to increment running cost
            input_tokens = message.usage.input_tokens
            output_tokens = message.usage.output_tokens
            
            call_cost = (input_tokens * HAIKU_INPUT_COST_PER_TOKEN) + (output_tokens * HAIKU_OUTPUT_COST_PER_TOKEN)
            
            # If this finished call pushes us over budget for subsequent items, or if we want to stop
            total_spent += call_cost
            
            # Extract text content safely from the block elements
            response_text = "".join([block.text for block in message.content if hasattr(block, 'text')])
            responses.append(response_text.strip())
            
        except Exception:
            # Break cleanly on API failures or network problems to protect structural loops
            break
            
    return responses

# Execute and store the results from the target invocation
results = run_prompts(SAMPLE_PROMPTS, 0.02)

<details>
<summary>💡 Hints: Requirements-to-Code Translation </summary>

| Requirement | Code Translation / Component | Why it works |
| :--- | :--- | :--- |
| **Accepts a list of prompts and a shared cost cap in USD.** | `def run_prompts(prompts: list[str], budget_usd: float)` | Standard Python typing explicitly maps the collection of text blocks to a list and the financial constraint to a floating-point number. |
| **Uses `MessagesClient` with `haiku_current`.** | `client.messages.create(model="claude-3-5-haiku-20241022", ...)` | In the `anthropic` SDK, `client.messages` is an instance of the `MessagesClient` class. The string representation for the current `haiku` model is explicitly declared via the model parameter. |
| **Calls each prompt in order.** | `for prompt in prompts:` | A basic `for` loop guarantees sequential execution. It processes each string item in the list sequentially from index `0` to the end, ensuring synchronous execution. |
| **Stops (without raising) if the next call would exceed `budget_usd`.** | `if total_spent >= budget_usd: break` and `try... except: break` | Using a `break` statement exits the execution loop cleanly without raising a crashing exception. The logic computes token overhead against a tracking balance (`total_spent`) after each call. |
| **Returns a list of response text strings.** | `responses.append(response_text.strip())` followed by `return responses` | An internal collection list captures valid string outputs parsed from Anthropic's unstructured `message.content` array blocks and returns them seamlessly at completion. |

</details>

In [15]:
print(results)

['Whale.', 'Jupiter.', 'Python', 'Brazil']


<hr style="border: 2px solid#003262;" />

## Q2: Streaming with Token Counting

Modify `run_prompts` from Q1 to use [`StreamingClient`](https://platform.claude.com/docs/en/build-with-claude/streaming) instead of [`MessagesClient`](https://platform.claude.com/docs/en/build-with-claude/working-with-messages). Collect each streamed chunk and count the total number of tokens streamed across all prompts.

* `client.messages.stream()` is a real-time streaming request. It immediately opens a server-sent events (SSE) connection and yields smaller chunks of text as they are being generated.

Store the total output token count in `total_tokens` (int).

Hint: Use `sc.last_call_tokens` -- that returns `(input_tokens, output_tokens)` after each stream ends.

In [17]:
# Q2: Streaming with token counting

# Current pricing for Claude 4.5 Haiku per million tokens (as of late 2024 / 2026 conventions)
# Input: $0.80 / million tokens ($0.0000008 per token)
# Output: $4.00 / million tokens ($0.0000040 per token)
HAIKU_INPUT_COST_PER_TOKEN = 0.80 / 1_000_000
HAIKU_OUTPUT_COST_PER_TOKEN = 4.00 / 1_000_000

# Global tracker for total output tokens streamed across all processed prompts
total_tokens = 0

def run_prompts(prompts: list[str], budget_usd: float) -> list[str]:
    global total_tokens

    # Initialize the client (assumes ANTHROPIC_API_KEY is in the environment)
    client = anthropic.Anthropic()
    
    responses = []
    total_spent = 0.0
    total_tokens = 0  # Reset for this execution block

    for prompt in prompts:
        # Check if we have any budget remaining before attempting a call
        if total_spent >= budget_usd:
            break
            
        try:
            # Using the stream context manager
            with client.messages.stream(
                model=haiku_current,
                max_tokens=50,
                messages=[{"role": "user", "content": prompt}]
            ) as stream:
                
                # Consume the stream to accumulate the chunks
                # stream.text_stream yields individual text snippets as they arrive
                collected_chunks = []
                for text in stream.text_stream:
                    collected_chunks.append(text)
                
                # Combine fragments and save to responses
                full_response = "".join(collected_chunks).strip()
                responses.append(full_response)
                
                # Fetch final token metrics after stream closure
                final_message = stream.get_final_message()

                # Calculate call cost
                input_tokens = final_message.usage.input_tokens
                output_tokens = final_message.usage.output_tokens              
                call_cost = (input_tokens * HAIKU_INPUT_COST_PER_TOKEN) + (output_tokens * HAIKU_OUTPUT_COST_PER_TOKEN)
                

                # Calculate accumulated streaming costs
                total_spent += call_cost
                total_tokens += output_tokens
                
        except Exception:
            break
            
    return responses, total_tokens, total_spent

# Execute and store the results from the target invocation
results = run_prompts(SAMPLE_PROMPTS, 0.02)

In [18]:
# ([responses], total_tokens, total_spent)
print(results)

(['Whale', 'Jupiter', 'Python', 'Brazil.'], 18, 0.0001272)


<hr style="border: 2px solid#003262;" />

## Q3: Realtime vs Batch Decision Justification

For each scenario below, decide whether to use the realtime Messages API or the Message Batches API. Justify each choice in a sentence.

| Scenario | Decision | Justification |
|---|---|---|
| A) A chatbot that answers user questions live in a web app | | |
| B) Nightly summarization of 10,000 log entries | | |
| C) A spell-checker that highlights errors as the user types | | |
| D) Generating weekly marketing copy for 500 product SKUs | | |


**YOUR ANSWERS HERE** - edit this cell to complete the table above.

| Scenario | Decision | Justification |
|---|---|---|
| A | | |
| B | | |
| C | | |
| D | | |

<hr style="border: 2px solid#003262;" />

## Q4: Batches API Round-Trip over Wikipedia Snippets

Using the fixture at `assets/homework_wikipedia_snippets.json` (10 article snippets), submit a Batches API job that asks Claude to summarize each snippet in one sentence.

1. Load the snippets from the JSON file.
2. Build one `BatchRequest` per snippet. Use the snippet's `id` as `custom_id`.
3. Submit the batch and store the batch handle in `batch_handle`.
4. (Optional - costs money) Poll to completion and store the results in `batch_results`.

In dry-run mode: print the request shapes and store the batch ID as `"dry-run-batch-id"` in `batch_id`.

* The purpose of a dry run is **to test a system, script, or process without committing resources, executing permanent changes, spending money, or risking system failure**.

  > ```python
  >if DRY_RUN:
  >    print("\n--- Dry Run Request Shapes ---")
  >    print(f"Total requests staged: {len(requests)}")
  >    print(f"Sample Custom ID: {requests[0]['custom_id']}")
  >    print(f"Sample Content Shape: {requests[0]['params']['messages'][0]['content'][:60]}...")
  >    
  >    batch_id = "dry-run-batch-id"
  >    batch_handle = None
  >else:
  >    # Submit the actual batch to the Anthropic API
  >    batch_handle = client.beta.messages.batches.create(requests=requests)
  >    batch_id = batch_handle.id
  >    print(f"Successfully submitted batch job with ID: {batch_id}")

In [ ]:
# Q4: Batches API round-trip over Wikipedia snippets
snippets_file = Path("assets/homework_wikipedia_snippets.json")
snippets = json.loads(snippets_file.read_text(encoding="utf-8"))
print(f"Loaded {len(snippets)} snippets")

In [ ]:
# Batching 
from anthropic.types.beta.messages.batch_create_params import Request

# Function used to call Batches API and help us understand how it handles requests async
def execute_asynchronous_batch_job(
    data_items: list[dict], 
    model_name: str = haiku_current,
    is_dry_run: bool = True
) -> list[dict]:
    """
    Submits a batch of text requests to the Anthropic Batches API, 
    monitors processing lifecycle metrics, and compiles the final outputs.
    
    How the Batches API handles these calls:
    1. CONCURRENCY: Anthropic processes your request list in parallel.
    2. ORDER: The API does NOT preserve array sequence. It outputs rows as they complete.
    3. TIMELINE: The API offers a 24-hour SLA window with a 50% cost discount.
    """

    client = anthropic.Anthropic()
    staged_requests = []
    compiled_results = []
    
    # --- COMPILING SPECIFICATION-ENFORCED PAYLOADS ---
    for item in data_items:
        # custom_id acts as a correlation spine to trace results back to source rows
        record_id = str(item.get("id"))
        text_content = item.get("text", "")
        
        # Enforce the structural schema required by the SDK type definitions
        request_param = Request(
            custom_id=record_id,
            params={
                "model": model_name,
                "max_tokens": 150,
                "messages": [
                    {
                        "role": "user",
                        "content": f"Summarize the following text in exactly one sentence:\n\n{text_content}"
                    }
                ]
            }
        )
        staged_requests.append(request_param)
        
    # --- LIFECYCLE MANAGEMENT (DRY RUN VS. SUBMISSION) ---
    if is_dry_run:
        print("\n=== [DRY RUN] BATCH SCHEMA VALIDATION ===")
        print(f" Staged Request Count: {len(staged_requests)}")
        print(f" Target Model Engine:  {model_name}")
        print(f" Structural Proof (First Item ID): {staged_requests[0]['custom_id']}")
        return [{"status": "dry_run_complete", "staged_count": len(staged_requests)}]
        
    print(f"\n⚡ Submitting {len(staged_requests)} records to the Asynchronous Processing Line...")
    # This single call drops off the entire array payload to the remote server architecture
    batch_job = client.beta.messages.batches.create(requests=staged_requests)
    target_batch_id = batch_job.id
    print(f" Job accepted. Assigned Batch Tracking Spine ID: {target_batch_id}")
    
    # --- STATE RE-EVALUATION & METADATA MONITORING ---
    print("\n⏳ Entering active monitoring state loop...")
    while True:
        # Polling explicitly fetches current system metrics from the API gateway
        status_report = client.beta.messages.batches.retrieve(target_batch_id)
        current_counts = status_report.request_counts
        
        print(
            f"[{time.strftime('%X')}] Status: {status_report.processing_status.upper()} | "
            f"Active: {current_counts.processing} | "
            f"Succeeded: {current_counts.succeeded} | "
            f"Failed: {current_counts.errored}"
        )
        
        if status_report.processing_status == "ended":
            print(f"\n Batch processing completed natively.")
            print(f" - Created At Timestamp: {status_report.created_at}")
            print(f" - Ended At Timestamp:   {status_report.ended_at}")
            break
        elif status_report.processing_status in ["canceling", "canceled"]:
            print(f"\n⚠ Job aborted by platform controls at: {status_report.cancel_initiated_at}")
            return []
            
        time.sleep(15)  # Safe buffer matching rate-limit guidelines
        
    # --- DISORDERED RESULT RETRIEVAL STREAMING ---
    print("\n Reading and resolving completed outputs from processing pipeline stream...")
    # Results are pulled down as a non-deterministic JSONL generator stream
    for final_record in client.beta.messages.batches.results(target_batch_id):
        compiled_results.append({
            "correlation_id": final_record.custom_id,
            "api_outcome": final_record.result
        })
        
    return compiled_results

In [24]:
# Example 1: Executing a safe Dry Run validation test
mock_data = [{"id": "wiki_01", "text": "Python is a high-level programming language."}]
dry_run_summary = execute_asynchronous_batch_job(mock_data, is_dry_run=True)


=== [DRY RUN] BATCH SCHEMA VALIDATION ===
 Staged Request Count: 1
 Target Model Engine:  claude-haiku-4-5-20251001
 Structural Proof (First Item ID): wiki_01


In [ ]:
if DRY_RUN:
    print("DRY RUN: skipping this live API cell")
else:
    # Example 2: Triggering a live production execution run (Requires source variables)
    live_results = execute_asynchronous_batch_job(snippets, model_name=haiku_current, is_dry_run=False)


## Sources Consulted

- [Messages API Reference](https://platform.claude.com/docs/en/api/messages) - retrieved 2026-09-02; volatile
- [Message Batches API](https://platform.claude.com/docs/en/api/creating-message-batches) - retrieved 2026-09-02; volatile
- [Streaming Guide](https://platform.claude.com/docs/en/docs/build-with-claude/streaming) - retrieved 2026-09-02; volatile

<hr style="border: 6px solid#003262;" />